# 3D ResNet DL Feature Extraction for ML

Extract final avgpool features from trained `resnet18_3D` models, apply PCA, MinMax-normalize PCA features, and save a ML-ready Excel table.

In [34]:
# ============================================================
# 1. Imports
# ============================================================

import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

sys.path.append('/host/d/Github/')

import Osteosarcoma.Build_lists.Build_list as Build_list
import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Image_3D.Generator_ResNet as Generator
import Osteosarcoma.Image_3D.resnet.model as model_module

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)


DEVICE: cuda


In [36]:
# ============================================================
# 2. Define dataset: use set123 random0 split, folds 0-6
# ============================================================

label = 'Prognosis'
label_col = label + '_label'
random_state = 0

patient_list_file = (
    '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/'
    f'image_label_info_set123_5fold_{label.lower()}_random{random_state}.xlsx'
)

data_root = '/host/e/D/Data/Habitats/Jishuitan/resampled_data_new'
out_root = '/host/d/projects/Habitats/radiomics/dl_3d_ml_allepoch10'
feature_np_out_dir = os.path.join(out_root, 'features_numpy')
case_feature_cache_root = os.path.join(feature_np_out_dir, 'case_features')

ff.make_folder([out_root, feature_np_out_dir, case_feature_cache_root])

CV_FOLDS = [0, 1, 2, 3, 4]
INTERNAL_TEST_FOLD = 5
EXTERNAL_TEST_FOLD = 6
ALL_FOLDS = CV_FOLDS + [INTERNAL_TEST_FOLD, EXTERNAL_TEST_FOLD]

print('patient_list_file:', patient_list_file)
print('data_root:', data_root)
print('out_root:', out_root)
print('case_feature_cache_root:', case_feature_cache_root)

build = Build_list.Build(patient_list_file)


def build_case_df(batch_list):
    fold_list, patient_set_list, patient_index_list, label_list, _, _ = build.__build__(
        batch_list=batch_list,
        label_column_name=label_col,
    )

    rows = []
    for i in range(len(patient_index_list)):
        patient_set = str(patient_set_list[i])
        patient_index = str(patient_index_list[i])
        image_path = os.path.join(data_root, patient_set, patient_index, 'img.nii.gz')
        mask_path = os.path.join(data_root, patient_set, patient_index, 'label.nii.gz')
        bbox_path = os.path.join(data_root, patient_set, patient_index, 'bbox_mask.nii.gz')
        rows.append({
            'Patient_set': patient_set,
            'Patient_index': patient_index,
            'fold': int(fold_list[i]),
            'Label': int(label_list[i]),
            'Image_filepath': image_path,
            'Mask_filepath': mask_path,
            'BBox_filepath': bbox_path,
        })
    return pd.DataFrame(rows)

fold_case_dfs = {fold: build_case_df([fold]) for fold in ALL_FOLDS}
all_case_df = pd.concat([fold_case_dfs[fold] for fold in ALL_FOLDS], ignore_index=True)

print('All cases:', all_case_df.shape)
print(all_case_df.groupby('fold')['Label'].agg(['count', 'mean']))
all_case_df.head()


patient_list_file: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_random0.xlsx
data_root: /host/e/D/Data/Habitats/Jishuitan/resampled_data_new
out_root: /host/d/projects/Habitats/radiomics/dl_3d_ml_allepoch10
case_feature_cache_root: /host/d/projects/Habitats/radiomics/dl_3d_ml_allepoch10/features_numpy/case_features
All cases: (351, 7)
      count      mean
fold                 
0        38  0.315789
1        38  0.289474
2        38  0.289474
3        37  0.297297
4        37  0.297297
5        98  0.295918
6        65  0.307692


,Patient_set,Patient_index,fold,Label,Image_filepath,Mask_filepath,BBox_filepath
0,set_1,11,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
1,set_1,36,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
2,set_1,40,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
3,set_1,50,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
4,set_1,51,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...


In [37]:
# ============================================================
# 3. Manually define which model is used for each fold
# ============================================================
# Fill these paths before running feature extraction.
# If you use one all-data 3D model for all cases, put the same checkpoint path
# for fold 0-4. Fold 5 can later use one selected model or the average of five.

model_depth = 18
in_channels = 3
image_size = (96, 96, 64)
batch_size = 4

# for dl_3d_ml_cv
# fold_model_paths = {
#     0: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_fold0/models/model-55.pt',
#     1: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_fold1/models/model-55.pt',
#     2: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_fold2/models/model-85.pt',
#     3: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_fold3/models/model-45.pt',
#     4: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_fold4/models/model-25.pt',
# }

# for dl_3d_ml_all, we use epoch 12
fold_model_paths = {
    0: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-12.pt',
    1: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-12.pt',
    2: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-12.pt',
    3: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-12.pt',
    4: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-12.pt',
}

for fold, path in fold_model_paths.items():
    print(f'fold{fold}_model_path:', path)


fold0_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt
fold1_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt
fold2_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt
fold3_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt
fold4_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt


In [38]:
# ============================================================
# 3b. Manually define models for internal/external test features
# ============================================================
# CV folds 0-4 still use fold_model_paths above.
# Internal/external test can use one model or multiple models.
# If one path is provided, that model's feature is used directly.
# If multiple paths are provided, the final feature is the average feature
# across those models. Per-model intermediate features are not saved.

# for dl_3d_ml_cv
# internal_test_model_paths = [
#     '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_fold5/models/model-60.pt',]

# external_test_model_paths = [
#     '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_fold6/models/model-35.pt',]


# for dl_3d_ml_all
internal_test_model_paths = [
    '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-12.pt',]

external_test_model_paths = [
    '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-12.pt',]

print('Internal test model paths:')
for path in internal_test_model_paths:
    print(' ', path)

print('External test model paths:')
for path in external_test_model_paths:
    print(' ', path)


Internal test model paths:
  /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt
External test model paths:
  /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt


In [39]:
# ============================================================
# Helper functions: model loading, avgpool feature extraction, and per-case cache
# ============================================================

def load_resnet_checkpoint(model, checkpoint_path):
    if checkpoint_path is None or str(checkpoint_path).strip() == '':
        raise ValueError('checkpoint_path is empty. Please fill model paths first.')
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(checkpoint_path)

    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint

    cleaned = {}
    for key, value in state_dict.items():
        if key.startswith('module.'):
            key = key[len('module.'):]
        cleaned[key] = value

    model.load_state_dict(cleaned, strict=True)
    return model


def build_loaded_model(checkpoint_path):
    model = model_module.build_resnet3d_model(
        model_depth=model_depth,
        num_classes=2,
        in_channels=in_channels,
    )
    model = load_resnet_checkpoint(model, checkpoint_path)
    model.to(DEVICE)
    model.eval()
    return model


def resnet3d_avgpool_features(model, x):
    """Return final adaptive-avgpool vector from the 3D ResNet backbone."""
    return model.forward_features(x)


def make_dataset(case_df):
    return Generator.Dataset_3D(
        patient_set_list=case_df['Patient_set'].astype(str).tolist(),
        patient_index_list=case_df['Patient_index'].astype(str).tolist(),
        x_file_list=case_df['Image_filepath'].astype(str).tolist(),
        y_list=case_df['Label'].astype(int).tolist(),
        data_root=data_root,
        target_image_size=image_size,
        normalize_factor='medicalnet',
        only_tumor_pixels='seg',
        augment_context='simple',
        shuffle=False,
        augment=False,
        augment_frequency=0,
    )


def extract_features_for_cases(model, case_df):
    dataset = make_dataset(case_df)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    features = []
    labels = []

    model.eval()
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE)
            feat = resnet3d_avgpool_features(model, batch_x)
            features.append(feat.detach().cpu().numpy())
            labels.extend(batch_y.cpu().numpy().astype(int).tolist())

    features = np.concatenate(features, axis=0) if len(features) > 0 else np.zeros((0, 0), dtype=np.float32)
    labels = np.asarray(labels).astype(int)

    if len(case_df) != features.shape[0]:
        raise RuntimeError(f'Feature number mismatch: cases={len(case_df)}, features={features.shape[0]}')

    return features.astype(np.float32), labels


def sanitize_case_id(patient_set, patient_index):
    return f'{str(patient_set)}_{str(patient_index)}'.replace('/', '_').replace(' ', '_')


def get_case_feature_path(split_name, row):
    cache_dir = os.path.join(case_feature_cache_root, split_name)
    os.makedirs(cache_dir, exist_ok=True)
    case_id = sanitize_case_id(row['Patient_set'], row['Patient_index'])
    return os.path.join(cache_dir, f'{case_id}_DLfeature.npy')


def extract_features_with_model_paths(case_df, model_paths, split_name):
    """Extract final DL features with per-case caching.

    If a case-level final feature file already exists, it is loaded and the case
    is skipped. If one model path is provided, that model's feature is saved.
    If multiple model paths are provided, the average feature is saved. Per-model
    intermediate features are intentionally not saved.
    """
    if not isinstance(model_paths, (list, tuple)) or len(model_paths) == 0:
        raise ValueError(f'{split_name}: model_paths must be a non-empty list or tuple.')

    case_df = case_df.copy().reset_index(drop=True)
    final_features = [None] * len(case_df)
    missing_positions = []
    missing_paths = []

    for row_i, row in case_df.iterrows():
        feature_path = get_case_feature_path(split_name, row)
        if os.path.isfile(feature_path):
            final_features[row_i] = np.load(feature_path).reshape(-1).astype(np.float32)
            print(f'  [{split_name}] cached feature found, skipping case:', row['Patient_set'], row['Patient_index'])
        else:
            missing_positions.append(row_i)
            missing_paths.append(feature_path)

    if len(missing_positions) > 0:
        missing_df = case_df.iloc[missing_positions].reset_index(drop=True)
        model_feature_list = []

        for model_i, model_path in enumerate(model_paths):
            print(f'  [{split_name}] extracting {len(missing_df)} missing cases with model {model_i + 1}/{len(model_paths)}')
            print('   ', model_path)
            model = build_loaded_model(model_path)
            features, labels_check = extract_features_for_cases(model, missing_df)

            expected_labels = missing_df['Label'].astype(int).to_numpy()
            if not np.array_equal(labels_check, expected_labels):
                raise RuntimeError(f'Label mismatch during {split_name} extraction with model {model_i}.')

            model_feature_list.append(features)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if len(model_feature_list) == 1:
            missing_features = model_feature_list[0]
        else:
            missing_features = np.mean(np.stack(model_feature_list, axis=0), axis=0).astype(np.float32)

        for local_i, row_i in enumerate(missing_positions):
            feature_1d = missing_features[local_i].astype(np.float32).reshape(-1)
            np.save(missing_paths[local_i], feature_1d)
            final_features[row_i] = feature_1d

    if any(feature is None for feature in final_features):
        raise RuntimeError(f'{split_name}: some case features were not filled.')

    final_features = np.stack(final_features, axis=0).astype(np.float32)
    return final_features


In [40]:
# ============================================================
# 4. Extract DL features for folds 0-4 using their corresponding models
# ============================================================

fold01234_feature_list = []
fold01234_meta_list = []

for fold in CV_FOLDS:
    print('\n============================================================')
    print('Extracting CV fold:', fold)
    case_df = fold_case_dfs[fold].copy().reset_index(drop=True)

    features = extract_features_with_model_paths(
        case_df=case_df,
        model_paths=[fold_model_paths[fold]],
        split_name=f'fold{fold}_cv',
    )

    fold01234_feature_list.append(features)
    fold01234_meta_list.append(case_df)

    np.save(os.path.join(feature_np_out_dir, f'fold{fold}_DLfeature.npy'), features)
    case_df.to_excel(os.path.join(feature_np_out_dir, f'fold{fold}_metadata.xlsx'), index=False)

fold01234_DLfeature = np.concatenate(fold01234_feature_list, axis=0)
fold01234_metadata = pd.concat(fold01234_meta_list, ignore_index=True)

np.save(os.path.join(feature_np_out_dir, 'fold01234_DLfeature.npy'), fold01234_DLfeature)
fold01234_metadata.to_excel(os.path.join(feature_np_out_dir, 'fold01234_metadata.xlsx'), index=False)

print('fold01234 feature shape:', fold01234_DLfeature.shape)
fold01234_metadata.head()



Extracting CV fold: 0
  [fold0_cv] extracting 38 missing cases with model 1/1
    /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt

Extracting CV fold: 1
  [fold1_cv] extracting 38 missing cases with model 1/1
    /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt

Extracting CV fold: 2
  [fold2_cv] extracting 38 missing cases with model 1/1
    /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt

Extracting CV fold: 3
  [fold3_cv] extracting 37 missing cases with model 1/1
    /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt

Extracting CV fold: 4
  [fold4_cv] extracting 37 missing cases with model 1/1
    /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64

,Patient_set,Patient_index,fold,Label,Image_filepath,Mask_filepath,BBox_filepath
0,set_1,11,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
1,set_1,36,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
2,set_1,40,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
3,set_1,50,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
4,set_1,51,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...


In [41]:
# ============================================================
# 5. Extract DL features for internal and external test
# ============================================================

fold5_df = fold_case_dfs[INTERNAL_TEST_FOLD].copy().reset_index(drop=True)
fold5_DLfeature = extract_features_with_model_paths(
    case_df=fold5_df,
    model_paths=internal_test_model_paths,
    split_name='fold5_internal_test',
)

np.save(os.path.join(feature_np_out_dir, 'fold5_DLfeature.npy'), fold5_DLfeature)
fold5_df.to_excel(os.path.join(feature_np_out_dir, 'fold5_metadata.xlsx'), index=False)

print('fold5 internal-test feature shape:', fold5_DLfeature.shape)

fold6_df = fold_case_dfs[EXTERNAL_TEST_FOLD].copy().reset_index(drop=True)
fold6_DLfeature = extract_features_with_model_paths(
    case_df=fold6_df,
    model_paths=external_test_model_paths,
    split_name='fold6_external_test',
)

np.save(os.path.join(feature_np_out_dir, 'fold6_DLfeature.npy'), fold6_DLfeature)
fold6_df.to_excel(os.path.join(feature_np_out_dir, 'fold6_metadata.xlsx'), index=False)

print('fold6 external-test feature shape:', fold6_DLfeature.shape)


  [fold5_internal_test] extracting 98 missing cases with model 1/1
    /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt
fold5 internal-test feature shape: (98, 512)
  [fold6_external_test] extracting 65 missing cases with model 1/1
    /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-10.pt
fold6 external-test feature shape: (65, 512)


In [42]:
# ============================================================
# 6. Feature extraction summary
# ============================================================

print('CV feature shape:', fold01234_DLfeature.shape)
print('Internal-test feature shape:', fold5_DLfeature.shape)
print('External-test feature shape:', fold6_DLfeature.shape)
print('Internal-test model count:', len(internal_test_model_paths))
print('External-test model count:', len(external_test_model_paths))


CV feature shape: (188, 512)
Internal-test feature shape: (98, 512)
External-test feature shape: (65, 512)
Internal-test model count: 1
External-test model count: 1


In [43]:
# ============================================================
# 7. PCA on raw DL features, then MinMax normalize PCA features
# ============================================================

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import joblib
import os
import json
import numpy as np
import pandas as pd


# ============================================================
# 1. Combine CV, internal-test, and external-test features
# ============================================================

all_DLfeature_raw = np.concatenate(
    [
        fold01234_DLfeature,
        fold5_DLfeature,
        fold6_DLfeature,
    ],
    axis=0,
)

all_metadata = pd.concat(
    [
        fold01234_metadata,
        fold5_df,
        fold6_df,
    ],
    ignore_index=True,
)

print("All raw DL feature shape:", all_DLfeature_raw.shape)
print("All metadata shape:", all_metadata.shape)
print(all_metadata.groupby('fold')['Label'].agg(['count', 'mean']))


# ============================================================
# 2. PCA directly on raw DL features
# ============================================================

pca_n_components = 0.99

pca = PCA(
    n_components=pca_n_components,
    random_state=random_state,
)

all_DLfeature_pca_raw = pca.fit_transform(all_DLfeature_raw)

print("Raw PCA feature shape:", all_DLfeature_pca_raw.shape)
print("Explained variance ratio sum:", float(np.sum(pca.explained_variance_ratio_)))
print("Raw PCA min:", float(np.min(all_DLfeature_pca_raw)))
print("Raw PCA max:", float(np.max(all_DLfeature_pca_raw)))


# ============================================================
# 3. MinMax normalize PCA features for final ML input
# ============================================================

pca_minmax_scaler = MinMaxScaler(feature_range=(0, 1))

all_DLfeature_pca = pca_minmax_scaler.fit_transform(all_DLfeature_pca_raw)

print("Final normalized PCA feature shape:", all_DLfeature_pca.shape)
print("Final PCA feature min:", float(np.min(all_DLfeature_pca)))
print("Final PCA feature max:", float(np.max(all_DLfeature_pca)))


# ============================================================
# 4. Save raw feature, raw PCA, normalized PCA, PCA model, and scaler
# ============================================================

os.makedirs(feature_np_out_dir, exist_ok=True)

np.save(os.path.join(feature_np_out_dir, "all_DLfeature_raw_selected.npy"), all_DLfeature_raw)
np.save(os.path.join(feature_np_out_dir, "all_DLfeature_PCA_raw.npy"), all_DLfeature_pca_raw)
np.save(os.path.join(feature_np_out_dir, "all_DLfeature_PCA_minmax.npy"), all_DLfeature_pca)

joblib.dump(pca, os.path.join(feature_np_out_dir, "pca_model.joblib"))
joblib.dump(pca_minmax_scaler, os.path.join(feature_np_out_dir, "pca_minmax_scaler.joblib"))


# ============================================================
# 5. Save settings
# ============================================================

pca_info = {
    "label": label,
    "random_state": random_state,
    "patient_list_file": patient_list_file,
    "model_depth": model_depth,
    "in_channels": in_channels,
    "image_size": image_size,
    "fold_model_paths": fold_model_paths,
    "internal_test_model_paths": internal_test_model_paths,
    "external_test_model_paths": external_test_model_paths,
    "internal_test_model_count": len(internal_test_model_paths),
    "external_test_model_count": len(external_test_model_paths),
    "raw_feature_normalization": "None",
    "pca_input": "raw DL features",
    "pca_n_components": pca_n_components,
    "pca_output_components": int(all_DLfeature_pca.shape[1]),
    "pca_explained_variance_ratio_sum": float(np.sum(pca.explained_variance_ratio_)),
    "final_feature_normalization": "MinMaxScaler fitted on PCA features; final ML input is [0,1]",
    "case_feature_cache_root": case_feature_cache_root,
}

with open(os.path.join(feature_np_out_dir, "pca_info.json"), "w", encoding="utf-8") as f:
    json.dump(pca_info, f, indent=2)

print('Saved PCA artifacts to:', feature_np_out_dir)


All raw DL feature shape: (351, 512)
All metadata shape: (351, 7)
      count      mean
fold                 
0        38  0.315789
1        38  0.289474
2        38  0.289474
3        37  0.297297
4        37  0.297297
5        98  0.295918
6        65  0.307692
Raw PCA feature shape: (351, 20)
Explained variance ratio sum: 0.9903148412704468
Raw PCA min: -11.704273223876953
Raw PCA max: 20.035417556762695
Final normalized PCA feature shape: (351, 20)
Final PCA feature min: 0.0
Final PCA feature max: 1.0
Saved PCA artifacts to: /host/d/projects/Habitats/radiomics/dl_3d_ml_allepoch10/features_numpy


In [45]:
# ============================================================
# 9. Save final MinMax-normalized PCA DL features as Excel table
# ============================================================

feature_columns = [
    f'DL_feature_{i+1:03d}'
    for i in range(all_DLfeature_pca.shape[1])
]

feature_df = pd.DataFrame(
    all_DLfeature_pca,
    columns=feature_columns,
)

metadata_columns = [
    'Patient_set',
    'Patient_index',
    'Image_filepath',
    'Mask_filepath',
    'fold',
    'Label',
]

ml_table_df = pd.concat(
    [
        all_metadata[metadata_columns].reset_index(drop=True),
        feature_df.reset_index(drop=True),
    ],
    axis=1,
)

save_path = os.path.join(
    out_root,
    'dl_3d_features_PCA.xlsx',
)

ml_table_df.to_excel(save_path, index=False)

print('Saved final MinMax-normalized PCA 3D DL feature table:')
print(save_path)
print('Shape:', ml_table_df.shape)

feature_cols_check = [
    col for col in ml_table_df.columns
    if col.startswith('DL_feature_')
]

print('Final feature min:', float(ml_table_df[feature_cols_check].min().min()))
print('Final feature max:', float(ml_table_df[feature_cols_check].max().max()))

ml_table_df.head()


Saved final MinMax-normalized PCA 3D DL feature table:
/host/d/projects/Habitats/radiomics/dl_3d_ml_allepoch10/dl_3d_features_PCA.xlsx
Shape: (351, 26)
Final feature min: 0.0
Final feature max: 1.0


,Patient_set,Patient_index,Image_filepath,Mask_filepath,fold,Label,DL_feature_001,DL_feature_002,DL_feature_003,DL_feature_004,...,DL_feature_011,DL_feature_012,DL_feature_013,DL_feature_014,DL_feature_015,DL_feature_016,DL_feature_017,DL_feature_018,DL_feature_019,DL_feature_020
0,set_1,11,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,0,0.240749,0.455050,0.555243,0.280126,...,0.451340,0.260679,0.472445,0.917658,0.456921,0.042671,0.280142,0.655392,0.596876,0.520221
1,set_1,36,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,0,0.347104,0.213175,0.985466,0.685366,...,0.424605,0.361116,0.505795,0.405309,0.400860,0.265850,0.078990,0.152048,0.358518,0.207659
2,set_1,40,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,0,0.164474,0.175312,0.464521,0.676313,...,0.367721,0.437049,0.330808,0.643849,0.612884,0.499094,0.451511,0.544161,0.421926,0.196475
3,set_1,50,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,0,0.502940,0.397200,0.959605,0.484050,...,0.704750,0.514901,0.723790,0.704568,0.603170,0.252140,0.747069,0.464270,0.178978,0.326287
4,set_1,51,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,1,0.123537,0.444427,0.430347,0.840808,...,0.078000,0.811671,0.385487,0.517074,0.461056,0.301092,0.429195,0.569996,0.519388,0.486030
